# VIntentAgent fresh locked eval (v8, hybrid K=5)

Eval-only notebook for `fresh_test_locked_20260626`. It does not train or tune anything.

Expected Kaggle input files/folders:
- `vi_droidcall_fresh_test_locked_20260626.jsonl`
- `android_tools.json`
- `adapter_v8/adapter_model.safetensors` and `adapter_v8/adapter_config.json`
- optional: `final_config.json`

Enable GPU and internet on Kaggle unless the base Qwen and dense retriever models are already available as Kaggle inputs.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install "peft==0.13.2" "transformers==4.46.3" "sentence-transformers==3.3.1" accelerate safetensors

import json, math, os, re, time, unicodedata, hashlib, shutil, inspect
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig

BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
DENSE_MODEL = 'intfloat/multilingual-e5-small'
ALPHA = 0.7
TOP_K = 5
LOCK_DATE = '20260626'
OUT_DIR = Path('/kaggle/working') / f'fresh_test_locked_{LOCK_DATE}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
def find_file(name):
    roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    for root in roots:
        if not root.exists():
            continue
        hits = list(root.rglob(name))
        if hits:
            return hits[0]
    raise FileNotFoundError(name)

def find_adapter_dir():
    roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    for root in roots:
        if not root.exists():
            continue
        for cfg in root.rglob('adapter_config.json'):
            if cfg.parent.name == 'adapter_v8' or (cfg.parent / 'adapter_model.safetensors').exists():
                return cfg.parent
    raise FileNotFoundError('adapter_v8/adapter_config.json')

def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest().upper()

tools_path = find_file('android_tools.json')
eval_path = find_file('vi_droidcall_fresh_test_locked_20260626.jsonl')
adapter_dir = find_adapter_dir()


# PEFT compatibility: adapter was saved with a newer PEFT (0.18.x). Kaggle's
# stable PEFT pin may not accept newly added config keys, so make a sanitized
# working copy while leaving the uploaded dataset immutable.
adapter_load_dir = OUT_DIR / 'adapter_v8_compat'
if adapter_load_dir.exists():
    shutil.rmtree(adapter_load_dir)
shutil.copytree(adapter_dir, adapter_load_dir)
raw_adapter_config = json.loads((adapter_load_dir / 'adapter_config.json').read_text(encoding='utf-8'))
allowed_lora_keys = set(inspect.signature(LoraConfig.__init__).parameters) - {'self'}
# Keep PEFT discriminator keys as well; older PEFT accepts these through config loading.
allowed_lora_keys |= {'peft_type', 'task_type'}
compat_adapter_config = {k: v for k, v in raw_adapter_config.items() if k in allowed_lora_keys}
( adapter_load_dir / 'adapter_config.json').write_text(
    json.dumps(compat_adapter_config, ensure_ascii=False, indent=2), encoding='utf-8'
)
removed_config_keys = sorted(set(raw_adapter_config) - set(compat_adapter_config))

config_path = None
try:
    config_path = find_file('final_config.json')
except FileNotFoundError:
    pass

tools = json.loads(tools_path.read_text(encoding='utf-8'))
tool_by_name = {t['name']: t for t in tools}
eval_data = load_jsonl(eval_path)

print('tools:', len(tools), tools_path)
print('eval rows:', len(eval_data), eval_path)
print('eval sha256:', sha256(eval_path))
print('adapter:', adapter_dir)
print('adapter compat:', adapter_load_dir)
print('removed adapter_config keys:', removed_config_keys)
if config_path:
    print('config:', config_path)
    print(config_path.read_text(encoding='utf-8')[:1000])

assert len(tools) == 24
assert len(eval_data) == 126
assert sha256(eval_path) == '051C811625F0D22925EA60868BDCE03A67B291E34976636306DB9BF8F13387B4'

In [ ]:
# BM25 + dense hybrid retriever, matching the v8 frozen config.
def strip_accents(text):
    text = str(text).replace('đ', 'd').replace('Đ', 'D')
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

def normalize_text(value):
    return re.sub(r'\s+', ' ', strip_accents(str(value)).lower().strip())

def normalize_phone(value):
    return re.sub(r'\D', '', str(value))

def tok(text):
    return re.findall(r'[a-z0-9_@.+/*:-]+', strip_accents(text).lower())

def tool_text(t):
    return ' '.join([t['name'].replace('_', ' '), t.get('description_vi', ''), t.get('description_en', ''), ' '.join(t.get('arguments', {}).keys())])

class BM25:
    def __init__(self, tools, k1=1.5, b=0.75):
        self.tools = tools
        self.k1, self.b = k1, b
        self.docs = [tok(tool_text(t)) for t in tools]
        self.lengths = [len(d) for d in self.docs]
        self.avg_len = sum(self.lengths) / max(len(self.lengths), 1)
        self.tf = [Counter(d) for d in self.docs]
        df = Counter()
        for d in self.docs:
            df.update(set(d))
        n = len(self.docs)
        self.idf = {term: math.log(1 + (n - freq + 0.5) / (freq + 0.5)) for term, freq in df.items()}

    def scores(self, query):
        qterms = tok(query)
        rows = []
        for i, tf in enumerate(self.tf):
            score = 0.0
            for term in qterms:
                f = tf.get(term, 0)
                if f:
                    denom = f + self.k1 * (1 - self.b + self.b * self.lengths[i] / max(self.avg_len, 1))
                    score += self.idf.get(term, 0) * f * (self.k1 + 1) / denom
            rows.append({'tool': self.tools[i]['name'], 'score': score})
        return rows

bm25 = BM25(tools)
dense_model = SentenceTransformer(DENSE_MODEL)

def passage(t):
    return f"passage: {t['name'].replace('_',' ')}. {t.get('description_vi','')}. {t.get('description_en','')}. Parameters: {', '.join(t.get('arguments',{}).keys())}."

tool_embs = dense_model.encode([passage(t) for t in tools], normalize_embeddings=True, batch_size=24, show_progress_bar=False)

def dense_scores(query):
    q = dense_model.encode([f'query: {query}'], normalize_embeddings=True, show_progress_bar=False)[0]
    sims = (tool_embs @ q).tolist()
    return [{'tool': tools[i]['name'], 'score': float(sims[i])} for i in range(len(tools))]

def hybrid_search(query, top_k=TOP_K, alpha=ALPHA):
    b = {r['tool']: r['score'] for r in bm25.scores(query)}
    d = {r['tool']: r['score'] for r in dense_scores(query)}
    bv, dv = list(b.values()), list(d.values())
    b_min, b_r = min(bv), max(max(bv) - min(bv), 1e-9)
    d_min, d_r = min(dv), max(max(dv) - min(dv), 1e-9)
    combined = [{'tool': name, 'score': round(alpha * (d[name] - d_min) / d_r + (1 - alpha) * (b[name] - b_min) / b_r, 6)} for name in b]
    combined.sort(key=lambda x: (-x['score'], x['tool']))
    return combined[:top_k]

print(f'hybrid ready: alpha={ALPHA}, top_k={TOP_K}, tool_embs={tool_embs.shape}')

In [ ]:
SYSTEM_PROMPT = '''Bạn là bộ định tuyến công cụ Android chạy ngoại tuyến.
Chỉ trả về một JSON object, không giải thích.

Nếu yêu cầu đủ thông tin:
{"tool":"TOOL_NAME","arguments":{},"requires_confirmation":false}

Nếu thiếu thông tin:
{"tool":null,"arguments":{},"requires_confirmation":false,"status":"clarification","message":"..."}

Nếu không có công cụ phù hợp:
{"tool":null,"arguments":{},"requires_confirmation":false,"status":"unsupported"}

Không tự bịa tham số. Hành động có confirmation=true phải đặt requires_confirmation=true.
'''

ROBUST_SUFFIX = '''
Ưu tiên độ chính xác tham số:
- Không thêm argument nếu người dùng không nói rõ.
- Với báo thức/lịch, giữ đúng giờ, phút, ngày lặp và chỉ thêm nhãn khi có nội dung nhắc.
- Với SMS/email, giữ nguyên người nhận, subject và body; không tóm tắt hoặc tự viết lại body.
- Phân biệt mở tài liệu lâu dài (ACTION_OPEN_DOCUMENT), lấy nội dung tạm thời (ACTION_GET_CONTENT), và tạo file mới (ACTION_CREATE_DOCUMENT).
- ACTION_IMAGE_CAPTURE dùng khi chụp ảnh ngay lập tức; INTENT_ACTION_STILL_IMAGE_CAMERA dùng khi mở ứng dụng camera.
- Loại bỏ khoảng trắng trong số điện thoại: "028 3823 4567" → "02838234567".
- Nếu thiếu người nhận, số điện thoại, email, nội dung, thời gian hoặc đối tượng cụ thể thì dùng tool null với status "clarification".
- Nếu yêu cầu nằm ngoài danh sách công cụ thì dùng tool null với status "unsupported".
'''

ROBUST_SYSTEM = SYSTEM_PROMPT + ROBUST_SUFFIX

def compact_tool(t):
    return {'name': t['name'], 'description': t.get('description_vi', ''), 'confirmation': t['confirmation'], 'arguments': t['arguments']}

def build_prompt(query):
    selected = [tool_by_name[r['tool']] for r in hybrid_search(query, top_k=TOP_K) if r['tool'] in tool_by_name]
    return json.dumps({'tools': [compact_tool(t) for t in selected], 'user_query': query}, ensure_ascii=False), [t['name'] for t in selected]

print('prompt ready')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True, torch_dtype=dtype, low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base, str(adapter_load_dir))
model = model.merge_and_unload().eval()
if torch.cuda.is_available():
    model = model.cuda()

print('model loaded')

In [ ]:
def extract_json(text):
    stripped = text.strip()
    try:
        return json.loads(stripped)
    except Exception:
        pass
    match = re.search(r'\{.*\}', stripped, flags=re.DOTALL)
    if not match:
        return {'tool': None, 'arguments': {}, 'requires_confirmation': False, 'status': 'rejected'}
    try:
        return json.loads(match.group(0))
    except Exception:
        return {'tool': None, 'arguments': {}, 'requires_confirmation': False, 'status': 'rejected'}

pred_path = OUT_DIR / 'v8_hybrid_k5_predictions.jsonl'
pred_path.unlink(missing_ok=True)

latencies = []
t_all = time.time()
with pred_path.open('w', encoding='utf-8') as f:
    for i, ex in enumerate(eval_data, 1):
        user_content, retrieved = build_prompt(ex['query'])
        messages = [{'role': 'system', 'content': ROBUST_SYSTEM}, {'role': 'user', 'content': user_content}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt')
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)
        decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        pred = extract_json(decoded)
        if pred.get('tool') in tool_by_name:
            pred['requires_confirmation'] = tool_by_name[pred['tool']]['confirmation']
        row = {'id': ex['id'], 'prediction': pred, 'raw_output': decoded, 'latency_ms': round(latency_ms, 3), 'retrieved_tools': retrieved, 'retriever': 'hybrid'}
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
        f.flush()
        exp_tool = ex['expected'].get('tool') or f"null({ex['expected'].get('status','')})"
        pred_tool = pred.get('tool') or f"null({pred.get('status','')})"
        ok = 'OK' if pred.get('tool') == ex['expected'].get('tool') else 'NO'
        print(f'[{i:3d}/{len(eval_data)}] {ok} {ex["id"]:<20} exp={exp_tool:<32} pred={pred_tool} {latency_ms/1000:.1f}s')

print(f'wrote {pred_path}')
print(f'total: {time.time() - t_all:.1f}s, mean generation latency: {np.mean(latencies)/1000:.2f}s')

In [ ]:
# Canonical evaluator matching prototype/src/evaluate.py.
HONORIFICS = {'anh','chi','chị','em','co','cô','chu','chú','bac','bác','ong','ông','ba','bà','thay','thầy','ban','bạn'}

def normalize_contact_name(text):
    cleaned = str(text).replace('_', ' ').strip()
    parts = cleaned.split(None, 1)
    if len(parts) == 2 and parts[0].lower() in HONORIFICS:
        return parts[1].strip()
    return cleaned

def type_matches(value, expected):
    checks = {
        'string': lambda x: isinstance(x, str),
        'integer': lambda x: isinstance(x, int) and not isinstance(x, bool),
        'boolean': lambda x: isinstance(x, bool),
        'array': lambda x: isinstance(x, list),
        'object': lambda x: isinstance(x, dict),
    }
    return expected in checks and checks[expected](value)

def validate(prediction):
    errors = []
    tool_name = prediction.get('tool')
    status = prediction.get('status')
    if tool_name is None:
        if status not in {'clarification', 'unsupported', 'rejected'}:
            errors.append('null tool requires clarification, unsupported or rejected status')
        return errors
    tool = tool_by_name.get(tool_name)
    if tool is None:
        return [f'unknown tool: {tool_name}']
    args = prediction.get('arguments')
    if not isinstance(args, dict):
        return ['arguments must be an object']
    schema = tool.get('arguments', {})
    unknown = sorted(set(args) - set(schema))
    if unknown:
        errors.append('unknown arguments: ' + ', '.join(unknown))
    for name, arg_schema in schema.items():
        if arg_schema.get('required') and (name not in args or args[name] is None):
            errors.append(f'missing required argument: {name}')
            continue
        if name not in args or args[name] is None:
            continue
        value = args[name]
        expected_type = arg_schema.get('type')
        if not type_matches(value, expected_type):
            errors.append(f'{name} must be {expected_type}')
            continue
        if 'enum' in arg_schema and value not in arg_schema['enum']:
            errors.append(f'{name} is outside enum')
        if expected_type == 'integer':
            if 'minimum' in arg_schema and value < arg_schema['minimum']:
                errors.append(f'{name} is below minimum')
            if 'maximum' in arg_schema and value > arg_schema['maximum']:
                errors.append(f'{name} is above maximum')
        if expected_type == 'array' and 'items' in arg_schema:
            if any(not type_matches(item, arg_schema['items']) for item in value):
                errors.append(f'{name} contains invalid item type')
    if prediction.get('requires_confirmation') != bool(tool.get('confirmation')):
        errors.append(f"requires_confirmation must be {str(bool(tool.get('confirmation'))).lower()}")
    return errors

def values_match(name, expected, actual):
    if isinstance(expected, dict):
        return isinstance(actual, dict) and all(k in actual and values_match(k, v, actual[k]) for k, v in expected.items())
    if isinstance(expected, list):
        if not isinstance(actual, list) or len(expected) != len(actual):
            return False
        return sorted(normalize_text(x) for x in expected) == sorted(normalize_text(x) for x in actual)
    if isinstance(expected, bool) or isinstance(expected, int):
        return expected == actual
    if 'phone' in name.lower():
        return normalize_phone(expected) == normalize_phone(actual)
    if name.lower() == 'name':
        return normalize_contact_name(normalize_text(expected)) == normalize_contact_name(normalize_text(actual))
    return normalize_text(expected) == normalize_text(actual)

def wilson(success, total, z=1.96):
    if total == 0:
        return 'N/A'
    p = success / total
    den = 1 + z*z/total
    centre = (p + z*z/(2*total)) / den
    margin = z * math.sqrt((p*(1-p) + z*z/(4*total))/total) / den
    return f'[{max(0, centre-margin):.3f}, {min(1, centre+margin):.3f}]'

pred_rows = load_jsonl(pred_path)
preds = {r['id']: r.get('prediction', r) for r in pred_rows}
c = Counter()
groups = defaultdict(Counter)
per_tool = defaultdict(Counter)
errors = []
per_example = []

for ex in eval_data:
    exp = ex['expected']
    pred = preds.get(ex['id'], {})
    expected_tool = exp.get('tool')
    predicted_tool = pred.get('tool')
    group = ex.get('group', 'unknown')
    tool_key = expected_tool or f"NULL:{exp.get('status')}"
    schema_errors = validate(pred)
    expected_args = exp.get('arguments', {})
    predicted_args = pred.get('arguments', {}) if isinstance(pred.get('arguments', {}), dict) else {}
    correct_args = sum(1 for k, v in expected_args.items() if k in predicted_args and values_match(k, v, predicted_args[k]))
    total_args = len(expected_args)
    exact_args = correct_args == total_args and set(predicted_args.keys()) == set(expected_args.keys())
    status_correct = True if expected_tool is not None else (predicted_tool is None and pred.get('status') == exp.get('status'))
    confirmation_correct = pred.get('requires_confirmation') == exp.get('requires_confirmation')
    tool_correct = expected_tool == predicted_tool
    e2e = tool_correct and exact_args and status_correct and confirmation_correct and not schema_errors
    for bucket in [c, groups[group], per_tool[tool_key]]:
        bucket['count'] += 1
        bucket['tool_correct'] += int(tool_correct)
        bucket['schema_valid'] += int(not schema_errors)
        bucket['argument_correct'] += correct_args
        bucket['argument_total'] += total_args
        bucket['confirmation_correct'] += int(confirmation_correct)
        bucket['status_correct'] += int(status_correct)
        bucket['end_to_end'] += int(e2e)
    rec = {'id': ex['id'], 'tool_correct': tool_correct, 'arguments_exact': exact_args, 'schema_valid': not bool(schema_errors), 'status_correct': status_correct, 'confirmation_correct': confirmation_correct, 'end_to_end': e2e}
    per_example.append(rec)
    if not e2e:
        errors.append({'id': ex['id'], 'query': ex['query'], 'expected': exp, 'prediction': pred, 'schema_errors': schema_errors})

def render_bucket(bucket):
    n = bucket['count']
    return {
        'count': int(n),
        'tool_selection_accuracy': bucket['tool_correct'] / n if n else 0,
        'schema_valid_rate': bucket['schema_valid'] / n if n else 0,
        'soft_argument_accuracy': bucket['argument_correct'] / bucket['argument_total'] if bucket['argument_total'] else 1.0,
        'requires_confirmation_accuracy': bucket['confirmation_correct'] / n if n else 0,
        'status_accuracy': bucket['status_correct'] / n if n else 0,
        'end_to_end_task_success': bucket['end_to_end'] / n if n else 0,
    }

report = render_bucket(c)
report['end_to_end_ci_95'] = wilson(c['end_to_end'], c['count'])
report['negative_clarification_accuracy'] = sum(1 for ex in eval_data if ex['expected'].get('tool') is None and preds.get(ex['id'], {}).get('status') == ex['expected'].get('status')) / sum(1 for ex in eval_data if ex['expected'].get('tool') is None)
report['groups'] = {k: render_bucket(v) for k, v in sorted(groups.items())}
report['per_tool'] = {k: render_bucket(v) for k, v in sorted(per_tool.items())}
report['errors'] = errors
report['per_example'] = per_example
report['fresh_eval_sha256'] = sha256(eval_path)
report['prediction_file'] = str(pred_path)
report['config'] = {'model': 'v8', 'base_model': BASE_MODEL, 'retriever': 'hybrid', 'top_k': TOP_K, 'alpha': ALPHA, 'robust_prompt': True, 'fix_confirmation_from_schema': True}

report_path = OUT_DIR / 'main_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({k: report[k] for k in ['count','tool_selection_accuracy','schema_valid_rate','soft_argument_accuracy','requires_confirmation_accuracy','negative_clarification_accuracy','end_to_end_task_success','end_to_end_ci_95']}, ensure_ascii=False, indent=2))
print('saved report:', report_path)

In [ ]:
summary_path = OUT_DIR / 'RUN_SUMMARY.md'
summary = f'''# Fresh locked eval summary

- Eval split: `{eval_path.name}`
- Eval SHA-256: `{sha256(eval_path)}`
- Model: v8 adapter + `{BASE_MODEL}`
- Retriever: hybrid K={TOP_K}, alpha={ALPHA}
- Prompt: SYSTEM_PROMPT + ROBUST_SUFFIX
- Predictions: `{pred_path.name}`
- Report: `main_report.json`

## Main metrics

- N: {report['count']}
- ToolAcc: {report['tool_selection_accuracy']:.4f}
- SchemaValid: {report['schema_valid_rate']:.4f}
- SoftArgAcc: {report['soft_argument_accuracy']:.4f}
- ConfirmationAcc: {report['requires_confirmation_accuracy']:.4f}
- Negative/clarification Acc: {report['negative_clarification_accuracy']:.4f}
- E2E: {report['end_to_end_task_success']:.4f} {report['end_to_end_ci_95']}
'''
summary_path.write_text(summary, encoding='utf-8')
print(summary)
print('Files in output dir:')
for p in sorted(OUT_DIR.iterdir()):
    print(p.name, p.stat().st_size)